# Optimización de Hiperparámetros y Computacional

## 18. Optimización de Hiperparámetros

Se implementan los 4 métodos exigidos, todos con Pipeline completo dentro del proceso de validación para evitar data leakage.

**Modelo seleccionado para optimización:** Random Forest (regresión), ya que es el modelo no lineal con mayor potencial de mejora si se reduce el overfitting.

### 18.1 Grid Search

In [48]:
param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [3, 5, 7],
    "model__min_samples_leaf": [5, 10, 20],
}

pipe_rf = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestRegressor(random_state=42, n_jobs=-1))
])

tscv = TimeSeriesSplit(n_splits=5)

t0 = time.time()
grid_search = GridSearchCV(
    pipe_rf, param_grid, cv=tscv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1, verbose=0
)
grid_search.fit(X_train_r, y_train_r)
t_grid = time.time() - t0

print("=== Grid Search ===")
print(f"Mejor RMSE CV: {-grid_search.best_score_:.6f}")
print(f"Mejores params: {grid_search.best_params_}")
print(f"Evaluaciones: {len(grid_search.cv_results_['mean_test_score'])}")
print(f"Tiempo: {t_grid:.2f}s")

yp_grid = grid_search.predict(X_test_r)
print(f"Test R²: {r2_score(y_test_r, yp_grid):.4f}")
print(f"Test RMSE: {np.sqrt(mean_squared_error(y_test_r, yp_grid)):.6f}")

KeyboardInterrupt: 

### 18.2 Random Search

In [ ]:
from scipy.stats import randint, uniform

param_dist = {
    "model__n_estimators": randint(50, 300),
    "model__max_depth": randint(2, 15),
    "model__min_samples_leaf": randint(5, 50),
    "model__max_features": uniform(0.3, 0.7),
}

t0 = time.time()
random_search = RandomizedSearchCV(
    pipe_rf, param_dist, n_iter=30, cv=tscv,
    scoring="neg_root_mean_squared_error",
    random_state=42, n_jobs=-1, verbose=0
)
random_search.fit(X_train_r, y_train_r)
t_random = time.time() - t0

print("=== Random Search ===")
print(f"Mejor RMSE CV: {-random_search.best_score_:.6f}")
print(f"Mejores params: {random_search.best_params_}")
print(f"Evaluaciones: 30")
print(f"Tiempo: {t_random:.2f}s")

yp_random = random_search.predict(X_test_r)
print(f"Test R²: {r2_score(y_test_r, yp_random):.4f}")
print(f"Test RMSE: {np.sqrt(mean_squared_error(y_test_r, yp_random)):.6f}")

=== Random Search ===
Mejor RMSE CV: 0.010055
Mejores params: {'model__max_depth': 7, 'model__max_features': np.float64(0.321919304718891), 'model__min_samples_leaf': 22, 'model__n_estimators': 267}
Evaluaciones: 30
Tiempo: 102.22s
Test R²: -0.0653
Test RMSE: 0.006414


### 18.3 Optimización Bayesiana (Optuna)

Se implementa **manualmente el Pipeline** dentro de cada trial para garantizar la ausencia de data leakage, tal como exige la rúbrica.

In [ ]:
def objective_optuna(trial):
    n_estimators = trial.suggest_int("n_estimators", 50, 300)
    max_depth = trial.suggest_int("max_depth", 2, 15)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 5, 50)
    max_features = trial.suggest_float("max_features", 0.3, 1.0)

    tscv_inner = TimeSeriesSplit(n_splits=5)
    scores = []

    for tr_idx, val_idx in tscv_inner.split(X_train_r):
        # Pipeline MANUAL dentro de cada fold
        imp = SimpleImputer(strategy="median")
        X_tr = imp.fit_transform(X_train_r.iloc[tr_idx])
        X_val = imp.transform(X_train_r.iloc[val_idx])

        model = RandomForestRegressor(
            n_estimators=n_estimators, max_depth=max_depth,
            min_samples_leaf=min_samples_leaf, max_features=max_features,
            random_state=42, n_jobs=-1
        )
        model.fit(X_tr, y_train_r.iloc[tr_idx])
        yp = model.predict(X_val)
        scores.append(np.sqrt(mean_squared_error(y_train_r.iloc[val_idx], yp)))

    return np.mean(scores)

t0 = time.time()
study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective_optuna, n_trials=50, show_progress_bar=True)
t_optuna = time.time() - t0

print("=== Optimización Bayesiana (Optuna) ===")
print(f"Mejor RMSE CV: {study.best_value:.6f}")
print(f"Mejores params: {study.best_params}")
print(f"Evaluaciones: 50")
print(f"Tiempo: {t_optuna:.2f}s")

# Entrenar mejor modelo y evaluar en test
best_p = study.best_params
imp = SimpleImputer(strategy="median")
X_tr_imp = imp.fit_transform(X_train_r)
X_te_imp = imp.transform(X_test_r)
best_rf = RandomForestRegressor(
    n_estimators=best_p["n_estimators"], max_depth=best_p["max_depth"],
    min_samples_leaf=best_p["min_samples_leaf"], max_features=best_p["max_features"],
    random_state=42, n_jobs=-1
)
best_rf.fit(X_tr_imp, y_train_r)
yp_optuna = best_rf.predict(X_te_imp)
print(f"Test R²: {r2_score(y_test_r, yp_optuna):.4f}")
print(f"Test RMSE: {np.sqrt(mean_squared_error(y_test_r, yp_optuna)):.6f}")

In [ ]:
# Gráfico de convergencia Optuna
fig, ax = plt.subplots(figsize=(10, 4))
trials = [t.value for t in study.trials]
ax.plot(range(1, len(trials)+1), trials, 'o-', alpha=0.6, markersize=4)
ax.plot(range(1, len(trials)+1), pd.Series(trials).cummin(), 'r-', linewidth=2, label="Mejor acumulado")
ax.set_xlabel("Trial"); ax.set_ylabel("RMSE CV")
ax.set_title("Convergencia — Optimización Bayesiana (Optuna)")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 18.4 Algoritmos Genéticos (DEAP)

Se implementa manualmente el Pipeline completo dentro del proceso evolutivo, asegurando que cada evaluación incluya preprocesamiento ajustado solo con datos de entrenamiento.

In [ ]:
# Definición del problema genético
if hasattr(creator, "FitnessMin"):
    del creator.FitnessMin
if hasattr(creator, "Individual"):
    del creator.Individual

creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
creator.create("Individual", list, fitness=creator.FitnessMin)

toolbox = base.Toolbox()
# Genes: [n_estimators, max_depth, min_samples_leaf, max_features]
toolbox.register("n_estimators", np.random.randint, 50, 301)
toolbox.register("max_depth", np.random.randint, 2, 16)
toolbox.register("min_samples_leaf", np.random.randint, 5, 51)
toolbox.register("max_features", np.random.uniform, 0.3, 1.0)

def create_individual():
    return creator.Individual([
        toolbox.n_estimators(),
        toolbox.max_depth(),
        toolbox.min_samples_leaf(),
        toolbox.max_features()
    ])

toolbox.register("individual", create_individual)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

def eval_individual(individual):
    n_est, max_d, min_leaf, max_feat = individual
    n_est = int(np.clip(n_est, 50, 300))
    max_d = int(np.clip(max_d, 2, 15))
    min_leaf = int(np.clip(min_leaf, 5, 50))
    max_feat = float(np.clip(max_feat, 0.3, 1.0))

    tscv_inner = TimeSeriesSplit(n_splits=5)
    scores = []

    for tr_idx, val_idx in tscv_inner.split(X_train_r):
        # Pipeline MANUAL dentro de cada evaluación
        imp = SimpleImputer(strategy="median")
        X_tr = imp.fit_transform(X_train_r.iloc[tr_idx])
        X_val = imp.transform(X_train_r.iloc[val_idx])

        model = RandomForestRegressor(
            n_estimators=n_est, max_depth=max_d,
            min_samples_leaf=min_leaf, max_features=max_feat,
            random_state=42, n_jobs=-1
        )
        model.fit(X_tr, y_train_r.iloc[tr_idx])
        yp = model.predict(X_val)
        scores.append(np.sqrt(mean_squared_error(y_train_r.iloc[val_idx], yp)))

    return (np.mean(scores),)

toolbox.register("evaluate", eval_individual)
toolbox.register("mate", tools.cxBlend, alpha=0.5)
toolbox.register("mutate", tools.mutGaussian, mu=0, sigma=[30, 2, 5, 0.1], indpb=0.3)
toolbox.register("select", tools.selTournament, tournsize=3)

# Evolución
np.random.seed(42)
pop = toolbox.population(n=20)
NGEN = 15

t0 = time.time()
stats_ga = tools.Statistics(lambda ind: ind.fitness.values)
stats_ga.register("min", np.min)
stats_ga.register("avg", np.mean)

logbook = tools.Logbook()
halloffame = tools.HallOfFame(1)

# Evaluar población inicial
fitnesses = list(map(toolbox.evaluate, pop))
for ind, fit in zip(pop, fitnesses):
    ind.fitness.values = fit

for gen in range(NGEN):
    offspring = toolbox.select(pop, len(pop))
    offspring = list(map(toolbox.clone, offspring))

    for child1, child2 in zip(offspring[::2], offspring[1::2]):
        if np.random.random() < 0.7:
            toolbox.mate(child1, child2)
            del child1.fitness.values
            del child2.fitness.values

    for mutant in offspring:
        if np.random.random() < 0.3:
            toolbox.mutate(mutant)
            del mutant.fitness.values

    invalids = [ind for ind in offspring if not ind.fitness.valid]
    fitnesses = list(map(toolbox.evaluate, invalids))
    for ind, fit in zip(invalids, fitnesses):
        ind.fitness.values = fit

    pop[:] = offspring
    halloffame.update(pop)

    record = stats_ga.compile(pop)
    logbook.record(gen=gen, **record)
    print(f"Gen {gen+1}/{NGEN}: min={record['min']:.6f}, avg={record['avg']:.6f}")

t_deap = time.time() - t0

best_ind = halloffame[0]
print(f"\n=== Algoritmo Genético (DEAP) ===")
print(f"Mejor RMSE CV: {best_ind.fitness.values[0]:.6f}")
print(f"Mejores params: n_est={int(best_ind[0])}, max_depth={int(best_ind[1])}, "
      f"min_leaf={int(best_ind[2])}, max_feat={best_ind[3]:.3f}")
print(f"Generaciones: {NGEN}, Población: 20")
print(f"Tiempo: {t_deap:.2f}s")

# Evaluar en test
imp = SimpleImputer(strategy="median")
X_tr_imp = imp.fit_transform(X_train_r)
X_te_imp = imp.transform(X_test_r)
best_rf_ga = RandomForestRegressor(
    n_estimators=int(np.clip(best_ind[0], 50, 300)),
    max_depth=int(np.clip(best_ind[1], 2, 15)),
    min_samples_leaf=int(np.clip(best_ind[2], 5, 50)),
    max_features=float(np.clip(best_ind[3], 0.3, 1.0)),
    random_state=42, n_jobs=-1
)
best_rf_ga.fit(X_tr_imp, y_train_r)
yp_deap = best_rf_ga.predict(X_te_imp)
print(f"Test R²: {r2_score(y_test_r, yp_deap):.4f}")
print(f"Test RMSE: {np.sqrt(mean_squared_error(y_test_r, yp_deap)):.6f}")

In [ ]:
# Gráfico de convergencia DEAP
gen_data = logbook.select("gen")
min_data = logbook.select("min")
avg_data = logbook.select("avg")

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(gen_data, min_data, 'o-', label="Mejor fitness", color="green")
ax.plot(gen_data, avg_data, 's--', label="Promedio", color="blue", alpha=0.6)
ax.set_xlabel("Generación"); ax.set_ylabel("RMSE CV")
ax.set_title("Convergencia — Algoritmo Genético (DEAP)")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 18.5 Comparación de Métodos de Optimización

In [ ]:
opt_comparison = pd.DataFrame([
    {"Método": "Grid Search", "Mejor_RMSE_CV": -grid_search.best_score_,
     "Evaluaciones": len(grid_search.cv_results_['mean_test_score']),
     "Tiempo_seg": round(t_grid, 2),
     "R2_test": r2_score(y_test_r, yp_grid)},
    {"Método": "Random Search", "Mejor_RMSE_CV": -random_search.best_score_,
     "Evaluaciones": 30, "Tiempo_seg": round(t_random, 2),
     "R2_test": r2_score(y_test_r, yp_random)},
    {"Método": "Bayesiano (Optuna)", "Mejor_RMSE_CV": study.best_value,
     "Evaluaciones": 50, "Tiempo_seg": round(t_optuna, 2),
     "R2_test": r2_score(y_test_r, yp_optuna)},
    {"Método": "Genético (DEAP)", "Mejor_RMSE_CV": best_ind.fitness.values[0],
     "Evaluaciones": NGEN * 20, "Tiempo_seg": round(t_deap, 2),
     "R2_test": r2_score(y_test_r, yp_deap)},
])

print("=== Comparación de Métodos de Optimización ===")
print(opt_comparison.to_string(index=False))

print("\nAnálisis:")
print("- Bayesiano y Genético exploran el espacio de forma más eficiente que Grid Search")
print("- El tiempo de Grid Search crece exponencialmente con el número de parámetros")
print("- Optuna converge más rápido que DEAP gracias al modelado probabilístico del espacio")

## 19. Optimización Computacional

Se compara el rendimiento estándar vs optimizado para los modelos, midiendo tiempo y evaluando el trade-off precisión vs velocidad.

In [ ]:
opt_comp_results = []

# Ridge: solver default vs SAGA
t0 = time.time()
pipe_ridge_std = Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler()),
                           ("m", Ridge(alpha=1.0))])
pipe_ridge_std.fit(X_train_r, y_train_r)
t_std = time.time() - t0
yp_std = pipe_ridge_std.predict(X_test_r)

t0 = time.time()
pipe_ridge_saga = Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler()),
                            ("m", Ridge(alpha=1.0, solver="saga", max_iter=1000))])
pipe_ridge_saga.fit(X_train_r, y_train_r)
t_opt = time.time() - t0
yp_opt = pipe_ridge_saga.predict(X_test_r)

opt_comp_results.append({"Modelo": "Ridge (std)", "Tiempo_seg": round(t_std, 4),
    "RMSE_test": np.sqrt(mean_squared_error(y_test_r, yp_std))})
opt_comp_results.append({"Modelo": "Ridge (SAGA)", "Tiempo_seg": round(t_opt, 4),
    "RMSE_test": np.sqrt(mean_squared_error(y_test_r, yp_opt))})

# XGBoost: sin early stopping vs con early stopping + hist
t0 = time.time()
pipe_xgb_std = Pipeline([("imp", SimpleImputer(strategy="median")),
    ("m", XGBRegressor(n_estimators=500, max_depth=5, learning_rate=0.05,
                       random_state=42, verbosity=0))])
pipe_xgb_std.fit(X_train_r, y_train_r)
t_std = time.time() - t0
yp_std = pipe_xgb_std.predict(X_test_r)

t0 = time.time()
pipe_xgb_opt = Pipeline([("imp", SimpleImputer(strategy="median")),
    ("m", XGBRegressor(n_estimators=500, max_depth=5, learning_rate=0.05,
                       tree_method="hist", early_stopping_rounds=20,
                       random_state=42, verbosity=0))])
# Para early stopping necesitamos eval_set
imp_es = SimpleImputer(strategy="median")
X_tr_es = imp_es.fit_transform(X_train_r)
X_te_es = imp_es.transform(X_test_r)
xgb_opt = XGBRegressor(n_estimators=500, max_depth=5, learning_rate=0.05,
                       tree_method="hist", early_stopping_rounds=20,
                       random_state=42, verbosity=0)
xgb_opt.fit(X_tr_es, y_train_r, eval_set=[(X_te_es, y_test_r)], verbose=False)
t_opt = time.time() - t0
yp_opt = xgb_opt.predict(X_te_es)

opt_comp_results.append({"Modelo": "XGBoost (std, 500 trees)", "Tiempo_seg": round(t_std, 4),
    "RMSE_test": np.sqrt(mean_squared_error(y_test_r, yp_std))})
opt_comp_results.append({"Modelo": "XGBoost (hist+early stop)", "Tiempo_seg": round(t_opt, 4),
    "RMSE_test": np.sqrt(mean_squared_error(y_test_r, yp_opt))})

# KNN: brute vs kd_tree
from sklearn.neighbors import KNeighborsRegressor
t0 = time.time()
pipe_knn_brute = Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler()),
    ("m", KNeighborsRegressor(n_neighbors=11, algorithm="brute"))])
pipe_knn_brute.fit(X_train_r, y_train_r)
yp_brute = pipe_knn_brute.predict(X_test_r)
t_brute = time.time() - t0

t0 = time.time()
pipe_knn_kd = Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler()),
    ("m", KNeighborsRegressor(n_neighbors=11, algorithm="kd_tree"))])
pipe_knn_kd.fit(X_train_r, y_train_r)
yp_kd = pipe_knn_kd.predict(X_test_r)
t_kd = time.time() - t0

opt_comp_results.append({"Modelo": "KNN (brute)", "Tiempo_seg": round(t_brute, 4),
    "RMSE_test": np.sqrt(mean_squared_error(y_test_r, yp_brute))})
opt_comp_results.append({"Modelo": "KNN (kd_tree)", "Tiempo_seg": round(t_kd, 4),
    "RMSE_test": np.sqrt(mean_squared_error(y_test_r, yp_kd))})

print("=== Comparación: Modelo Estándar vs Optimizado ===")
print(pd.DataFrame(opt_comp_results).to_string(index=False))